# Robust AI-Image Detector — Free Kaggle Training

This notebook clones the public repository, streams a 6,000-image balanced SID_Set subset (not the full 140 GB dataset), trains clean and robustness-aware models, evaluates both across the complete challenge transform grid, validates the final checkpoint, and packages only reproducibility artifacts. Enable a Kaggle GPU and internet access before running all cells.

The first code cell launches a real CUDA kernel before touching `/kaggle/working`. On a restart, an existing clone is fast-forwarded without deleting ignored files, and a complete validated 6,000-image subset is reused instead of downloaded again.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import torch

REPO_URL = "https://github.com/LINGSIHAN/TikTok-Hackathon-Track-5.git"
BRANCH = "master"
PROJECT_DIR = Path("/kaggle/working/TikTok-Hackathon-Track-5")
SUBSET_SIZE = 6_000
SEED = 42
DATASET_NAME = "saberzl/SID_Set"
DATASET_REVISION = "dc03ead57929879319ce30a82bfcfb8d317b10bd"
SOURCE_SPLIT = "train"
SHUFFLE_BUFFER = 512
SHA256_DEFINITION = "SHA-256 of the stored normalized JPEG bytes"
ALLOWED_GENERATED_UNTRACKED = {
    "data/processed/manifest.csv",
    "data/processed/manifest_summary.json",
    "artifacts/checkpoints/model.safetensors",
    "artifacts/checkpoints/model_metadata.json",
    "artifacts/metrics/training_history.json",
    "artifacts/metrics/metrics.json",
    "artifacts/metrics/predictions.csv",
    "artifacts/metrics/robustness.png",
    "artifacts/metrics/run_context.json",
    "artifacts/metrics/pip_freeze.txt",
    "artifacts/metrics/clean_baseline/metrics.json",
    "artifacts/metrics/clean_baseline/predictions.csv",
    "artifacts/metrics/clean_baseline/robustness.png",
}

def run(*args):
    print("+", " ".join(map(str, args)))
    subprocess.run([str(arg) for arg in args], check=True)

def cuda_recovery_hint(gpu_name, required_arch):
    if "P100" in gpu_name.upper() or required_arch == "sm_60":
        return (
            "Kaggle's current CUDA 12.8 PyTorch image omits Pascal/P100 "
            "sm_60 kernels. Preserve /kaggle/working, then choose T4 x2 "
            "in Settings (recommended), or install the official CUDA 12.6 "
            "fallback with `pip install --force-reinstall torch==2.9.1 "
            "torchvision==0.24.1 --index-url "
            "https://download.pytorch.org/whl/cu126` and restart the session."
        )
    return (
        "Choose a Kaggle GPU supported by this PyTorch build, or install a "
        "matching official PyTorch CUDA wheel before continuing."
    )

def validate_cuda_runtime():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "No GPU detected. In Kaggle choose Settings > Accelerator > GPU, "
            "then restart the session."
        )

    gpu_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    required_arch = f"sm_{capability[0]}{capability[1]}"
    compiled_arches = set(torch.cuda.get_arch_list())
    print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
    print("GPU:", gpu_name, "capability:", capability)
    print("Compiled CUDA architectures:", sorted(compiled_arches))

    if required_arch not in compiled_arches:
        raise RuntimeError(
            f"PyTorch was not compiled for {gpu_name} ({required_arch}). "
            + cuda_recovery_hint(gpu_name, required_arch)
        )

    try:
        probe = torch.arange(16, dtype=torch.float32, device="cuda").reshape(4, 4)
        probe_value = float((probe @ probe.T).sum().item())
        torch.cuda.synchronize()
    except Exception as error:
        raise RuntimeError(
            f"CUDA is visible but a real tensor kernel failed on {gpu_name}. "
            + cuda_recovery_hint(gpu_name, required_arch)
        ) from error

    print("CUDA tensor preflight passed; checksum:", probe_value)
    return {
        "gpu": gpu_name,
        "capability": list(capability),
        "required_arch": required_arch,
        "compiled_arches": sorted(compiled_arches),
        "cuda_runtime": torch.version.cuda,
    }

def validate_cuda_subprocess(python_executable, loaded_torch_version):
    probe_source = "\n".join([
        "import json",
        "import platform",
        "import torch",
        "info = {",
        "    'ok': False,",
        "    'python': platform.python_version(),",
        "    'pytorch': torch.__version__,",
        "    'cuda_runtime': torch.version.cuda,",
        "}",
        "try:",
        "    import torchvision",
        "    info['torchvision'] = torchvision.__version__",
        "    if not torch.cuda.is_available():",
        "        raise RuntimeError('CUDA is not available')",
        "    name = torch.cuda.get_device_name(0)",
        "    capability = torch.cuda.get_device_capability(0)",
        "    required_arch = f'sm_{capability[0]}{capability[1]}'",
        "    compiled_arches = sorted(torch.cuda.get_arch_list())",
        "    info.update(gpu=name, capability=list(capability), required_arch=required_arch, compiled_arches=compiled_arches)",
        "    if required_arch not in compiled_arches:",
        "        raise RuntimeError(f'PyTorch is missing compiled architecture {required_arch}')",
        "    probe = torch.arange(16, dtype=torch.float32, device='cuda').reshape(4, 4)",
        "    info['probe_checksum'] = float((probe @ probe.T).sum().item())",
        "    torch.cuda.synchronize()",
        "    info['ok'] = True",
        "except Exception as error:",
        "    info['error'] = f'{type(error).__name__}: {error}'",
        "print(json.dumps(info, sort_keys=True))",
        "raise SystemExit(0 if info['ok'] else 17)",
    ])
    completed = subprocess.run(
        [str(python_executable), "-c", probe_source],
        check=False,
        capture_output=True,
        text=True,
    )
    stdout_lines = [line for line in completed.stdout.splitlines() if line.strip()]
    try:
        info = json.loads(stdout_lines[-1])
    except (IndexError, json.JSONDecodeError) as error:
        raise RuntimeError(
            "Fresh training-interpreter CUDA probe returned no valid JSON. "
            "Restart the Kaggle session before continuing. stderr: "
            + completed.stderr[-2000:]
        ) from error
    fresh_torch_version = str(info.get("pytorch", "unknown"))
    if fresh_torch_version != loaded_torch_version:
        raise RuntimeError(
            "Dependency installation changed the installed PyTorch version from "
            f"{loaded_torch_version} to {fresh_torch_version}, while this notebook "
            "kernel still has the old version loaded. Preserve Files in "
            "/kaggle/working, restart the Kaggle session, then rerun the notebook "
            "from the first cell. Prepared data will be reused."
        )
    if completed.returncode != 0 or not info.get("ok"):
        gpu_name = str(info.get("gpu", "unknown GPU"))
        required_arch = str(info.get("required_arch", "unknown architecture"))
        raise RuntimeError(
            "The fresh interpreter that training will use failed its CUDA probe: "
            + str(info.get("error", "unknown error"))
            + ". "
            + cuda_recovery_hint(gpu_name, required_arch)
        )
    print("Fresh training-interpreter CUDA preflight passed:", json.dumps(info, sort_keys=True))
    return info

def sha256_file(path):
    hasher = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def inspect_prepared_subset(project_dir, *, verify_file_hashes=True):
    manifest_path = project_dir / "data/processed/manifest.csv"
    summary_path = project_dir / "data/processed/manifest_summary.json"
    if not manifest_path.is_file() or not summary_path.is_file():
        return False, "manifest or summary is missing"

    required_columns = {
        "path", "label", "split", "dataset", "source_split",
        "source_id", "sha256",
    }
    try:
        with manifest_path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            if reader.fieldnames is None or not required_columns.issubset(reader.fieldnames):
                return False, "manifest schema is incomplete"
            rows = list(reader)
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, csv.Error, json.JSONDecodeError) as error:
        return False, f"manifest cannot be read: {error}"

    if not isinstance(summary, dict):
        return False, "manifest summary must be a JSON object"
    if len(rows) != SUBSET_SIZE:
        return False, f"expected {SUBSET_SIZE} manifest rows"

    per_class = SUBSET_SIZE // 2
    per_class_targets = {
        "train": int(per_class * 0.8),
        "val": int(per_class * 0.1),
    }
    per_class_targets["test"] = per_class - sum(per_class_targets.values())
    expected_counts = {
        (split, str(label)): count
        for split, count in per_class_targets.items()
        for label in (0, 1)
    }
    expected_summary_split_counts = {
        split: {str(label): expected_counts[(split, str(label))] for label in (0, 1)}
        for split in ("train", "val", "test")
    }
    expected_summary_fields = {
        "schema_version": 1,
        "dataset": DATASET_NAME,
        "source_split": SOURCE_SPLIT,
        "seed": SEED,
        "total": SUBSET_SIZE,
        "sha256_definition": SHA256_DEFINITION,
        "class_counts": {"0_real": per_class, "1_full_synthetic": per_class},
        "split_counts": expected_summary_split_counts,
    }
    for field, expected_value in expected_summary_fields.items():
        if summary.get(field) != expected_value:
            return False, f"summary field {field!r} does not match the expected value"
    if summary.get("dataset_revision") not in (None, DATASET_REVISION):
        return False, "summary dataset revision does not match the pinned SID_Set revision"
    if summary.get("shuffle_buffer") not in (None, SHUFFLE_BUFFER):
        return False, "summary shuffle buffer does not match the selection contract"

    actual_counts = {key: 0 for key in expected_counts}
    hashes = set()
    source_splits = {}
    project_root = project_dir.resolve()

    for index, row in enumerate(rows, start=1):
        key = (row["split"], row["label"])
        if key not in actual_counts:
            return False, f"unexpected split/label pair: {key}"
        actual_counts[key] += 1
        if row["dataset"] != DATASET_NAME or row["source_split"] != SOURCE_SPLIT:
            return False, "manifest dataset/source_split does not match the run contract"
        digest = row["sha256"]
        if re.fullmatch(r"[0-9a-f]{64}", digest or "") is None:
            return False, f"invalid SHA-256 digest at manifest row {index}"
        if digest in hashes:
            return False, "duplicate SHA-256 value"
        hashes.add(digest)
        source_id = row["source_id"]
        if not source_id:
            return False, "empty source_id"
        previous_split = source_splits.setdefault(source_id, row["split"])
        if previous_split != row["split"]:
            return False, f"source_id crosses splits: {source_id}"
        relative_path = Path(row["path"])
        expected_directory = Path("data/raw") / ("authentic" if row["label"] == "0" else "generated")
        if relative_path.is_absolute() or relative_path.parent != expected_directory:
            return False, f"image path is outside the label directory: {row['path']}"
        if relative_path.name != f"{digest}.jpg":
            return False, f"image filename does not match its digest: {row['path']}"
        image_path = (project_dir / relative_path).resolve()
        try:
            image_path.relative_to(project_root)
        except ValueError:
            return False, f"image path escapes project: {row['path']}"
        if not image_path.is_file() or image_path.stat().st_size == 0:
            return False, f"missing or empty image: {row['path']}"
        if verify_file_hashes:
            try:
                actual_digest = sha256_file(image_path)
            except OSError as error:
                return False, f"cannot hash {row['path']}: {error}"
            if actual_digest != digest:
                return False, f"image content hash mismatch: {row['path']}"
            if index % 1000 == 0:
                print(f"Verified {index}/{SUBSET_SIZE} prepared image hashes.")

    if actual_counts != expected_counts:
        return False, f"unexpected split/class counts: {actual_counts}"
    validation_scope = "content hashes" if verify_file_hashes else "structure"
    return True, f"validated {len(rows)} images, manifest rows, summary, and {validation_scope}"

# This must pass before any existing project or prepared data is touched.
CUDA_INFO = validate_cuda_runtime()

In [ ]:
REUSE_PREPARED_DATA, PREPARED_DATA_STATUS = inspect_prepared_subset(PROJECT_DIR)
print("Prepared-data precheck:", PREPARED_DATA_STATUS)

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a Git repository; refusing to "
            "delete it because it may contain reusable data. Move it aside "
            "manually after backing it up."
        )
    origin_url = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    if origin_url.rstrip("/").removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError(f"Unexpected origin {origin_url}; refusing to update in place.")
    tracked_changes = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
        text=True,
    ).strip()
    if tracked_changes:
        raise RuntimeError(
            "Tracked changes exist in the Kaggle clone; refusing to overwrite them:\n"
            + tracked_changes
        )
    untracked = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "ls-files", "--others", "--exclude-standard"],
        text=True,
    ).splitlines()
    unexpected_untracked = sorted(set(untracked) - ALLOWED_GENERATED_UNTRACKED)
    if unexpected_untracked:
        raise RuntimeError(
            "Unexpected untracked files would make the run non-reproducible: "
            + ", ".join(unexpected_untracked)
        )
    print("Updating tracked code in place; ignored data and run artifacts are preserved.")
    run("git", "-C", PROJECT_DIR, "fetch", "--depth", "50", "origin", BRANCH)
    run("git", "-C", PROJECT_DIR, "merge", "--ff-only", "FETCH_HEAD")
    fetched_commit = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "FETCH_HEAD"], text=True
    ).strip()
    updated_head = subprocess.check_output(
        ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
    if updated_head != fetched_commit:
        raise RuntimeError(
            "The Kaggle clone is locally ahead of or different from fetched master; "
            "refusing to train an unverified commit. Back up the clone and restore "
            "it to origin/master manually."
        )
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, PROJECT_DIR)

post_update_reuse, post_update_status = inspect_prepared_subset(
    PROJECT_DIR, verify_file_hashes=False
)
if REUSE_PREPARED_DATA and not post_update_reuse:
    raise RuntimeError("Prepared data became invalid during the code-only update: " + post_update_status)
REUSE_PREPARED_DATA = REUSE_PREPARED_DATA and post_update_reuse
if REUSE_PREPARED_DATA:
    PREPARED_DATA_STATUS = post_update_status
os.chdir(PROJECT_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Training repository commit:", COMMIT)
print("Prepared-data status after code update:", PREPARED_DATA_STATUS)
KERNEL_TORCH_VERSION_BEFORE_PIP = torch.__version__
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt")

In [ ]:
# Probe a brand-new process using the exact interpreter that launches training.
FRESH_CUDA_INFO = validate_cuda_subprocess(
    sys.executable, KERNEL_TORCH_VERSION_BEFORE_PIP
)
CUDA_INFO = FRESH_CUDA_INFO

In [ ]:
run(sys.executable, "-m", "pytest", "-q")

In [ ]:
if REUSE_PREPARED_DATA:
    REUSE_PREPARED_DATA, PREPARED_DATA_STATUS = inspect_prepared_subset(
        PROJECT_DIR, verify_file_hashes=False
    )
    if not REUSE_PREPARED_DATA:
        raise RuntimeError("Prepared subset changed after validation: " + PREPARED_DATA_STATUS)
    print("Reusing prepared SID_Set subset:", PREPARED_DATA_STATUS)
else:
    print("No reusable prepared subset:", PREPARED_DATA_STATUS)
    run(
        sys.executable,
        "scripts/prepare_sid_subset.py",
        "--total", str(SUBSET_SIZE),
        "--seed", str(SEED),
        "--revision", DATASET_REVISION,
        "--shuffle-buffer", str(SHUFFLE_BUFFER),
    )
    REUSE_PREPARED_DATA, PREPARED_DATA_STATUS = inspect_prepared_subset(PROJECT_DIR)
    if not REUSE_PREPARED_DATA:
        raise RuntimeError("Prepared subset failed validation: " + PREPARED_DATA_STATUS)

summary_path = Path("data/processed/manifest_summary.json")
summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))
RECORDED_DATASET_REVISION = summary_payload.get("dataset_revision")
RECORDED_SHUFFLE_BUFFER = summary_payload.get("shuffle_buffer")
DATASET_REVISION_STATUS = (
    "pinned" if RECORDED_DATASET_REVISION == DATASET_REVISION else "legacy_unrecorded"
)
print("Dataset revision status:", DATASET_REVISION_STATUS)

import pandas as pd

manifest = pd.read_csv("data/processed/manifest.csv")
counts = manifest.groupby(["split", "label"]).size().unstack(fill_value=0)
print(counts)
assert len(manifest) == SUBSET_SIZE
assert set(manifest["label"]) == {0, 1}
assert set(manifest["split"]) == {"train", "val", "test"}
assert (counts > 0).all().all()
assert not manifest["sha256"].duplicated().any()
assert manifest.groupby("source_id")["split"].nunique().max() == 1
assert Path("data/processed/manifest_summary.json").is_file()

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_clean.yaml", "--device", "cuda")

In [ ]:
run(
    sys.executable, "-m", "src.evaluation.evaluate",
    "--manifest", "data/processed/manifest.csv",
    "--checkpoint", "artifacts/runs/clean/model.safetensors",
    "--split", "test",
    "--output-dir", "artifacts/metrics/clean_baseline",
    "--device", "cuda",
)

In [ ]:
run(sys.executable, "-m", "src.training.train", "--config", "configs/train_robust.yaml", "--device", "cuda")

In [ ]:
run(
    sys.executable, "-m", "src.evaluation.evaluate",
    "--manifest", "data/processed/manifest.csv",
    "--checkpoint", "artifacts/checkpoints/model.safetensors",
    "--split", "test",
    "--output-dir", "artifacts/metrics",
    "--device", "cuda",
)

In [ ]:
required = [
    Path("artifacts/checkpoints/model.safetensors"),
    Path("artifacts/checkpoints/model_metadata.json"),
    Path("artifacts/metrics/training_history.json"),
    Path("artifacts/metrics/metrics.json"),
    Path("artifacts/metrics/predictions.csv"),
    Path("artifacts/metrics/robustness.png"),
    Path("artifacts/runs/clean/model.safetensors"),
    Path("artifacts/runs/clean/model_metadata.json"),
    Path("artifacts/runs/clean/history.json"),
    Path("artifacts/metrics/clean_baseline/metrics.json"),
    Path("artifacts/metrics/clean_baseline/predictions.csv"),
    Path("artifacts/metrics/clean_baseline/robustness.png"),
    Path("data/processed/manifest.csv"),
    Path("data/processed/manifest_summary.json"),
]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise RuntimeError("Missing or empty required artifacts: " + ", ".join(missing))

for metrics_path in [Path("artifacts/metrics/metrics.json"), Path("artifacts/metrics/clean_baseline/metrics.json")]:
    payload = json.loads(metrics_path.read_text())
    assert len(payload["scenarios"]) == 20, f"Expected 20 scenarios in {metrics_path}"
    for scenario in payload["scenarios"]:
        assert scenario["num_samples"] == 600, f"Expected 600 test images in {metrics_path}"
        for name, value in scenario["metrics"].items():
            assert isinstance(value, (int, float)) and math.isfinite(value), f"Non-finite {name} in {metrics_path}"

from PIL import Image
from src.inference.predictor import Predictor

test_path = Path(manifest.loc[manifest["split"] == "test", "path"].iloc[0])
predictor = Predictor.from_checkpoint("artifacts/checkpoints/model.safetensors", device="cuda")
with Image.open(test_path) as image:
    smoke_probability = predictor.predict_pil(image)
assert 0.0 <= smoke_probability <= 1.0
print("Checkpoint smoke probability:", smoke_probability)

In [ ]:
export_dir = Path("/kaggle/working/export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True)

relative_files = [
    "artifacts/checkpoints/model.safetensors",
    "artifacts/checkpoints/model_metadata.json",
    "artifacts/runs/clean/model.safetensors",
    "artifacts/runs/clean/model_metadata.json",
    "artifacts/runs/clean/history.json",
    "artifacts/metrics/training_history.json",
    "artifacts/metrics/metrics.json",
    "artifacts/metrics/predictions.csv",
    "artifacts/metrics/robustness.png",
    "artifacts/metrics/clean_baseline/metrics.json",
    "artifacts/metrics/clean_baseline/predictions.csv",
    "artifacts/metrics/clean_baseline/robustness.png",
    "data/processed/manifest.csv",
    "data/processed/manifest_summary.json",
    "configs/train_clean.yaml",
    "configs/train_robust.yaml",
    "requirements.txt",
    "requirements-train.txt",
]
for relative in relative_files:
    source = Path(relative)
    destination = export_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

from importlib.metadata import version
from src.data.preprocessing import PREPROCESSING_CONTRACT_ID

run_context = {
    "repository": REPO_URL,
    "branch": BRANCH,
    "commit": COMMIT,
    "subset_size": SUBSET_SIZE,
    "seed": SEED,
    "shuffle_buffer": RECORDED_SHUFFLE_BUFFER,
    "dataset_revision": RECORDED_DATASET_REVISION,
    "dataset_revision_status": DATASET_REVISION_STATUS,
    "pytorch": CUDA_INFO["pytorch"],
    "torchvision": CUDA_INFO["torchvision"],
    "pillow": version("Pillow"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "datasets": version("datasets"),
    "huggingface_hub": version("huggingface_hub"),
    "safetensors": version("safetensors"),
    "python": CUDA_INFO["python"],
    "gpu": CUDA_INFO["gpu"],
    "cuda_runtime": CUDA_INFO["cuda_runtime"],
    "gpu_capability": CUDA_INFO["capability"],
    "required_cuda_arch": CUDA_INFO["required_arch"],
    "compiled_cuda_arches": CUDA_INFO["compiled_arches"],
    "cuda_probe_checksum": CUDA_INFO["probe_checksum"],
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "preprocessing_contract": PREPROCESSING_CONTRACT_ID,
    "manifest_sha256": hashlib.sha256(Path("data/processed/manifest.csv").read_bytes()).hexdigest(),
    "checkpoint_sha256": hashlib.sha256(Path("artifacts/checkpoints/model.safetensors").read_bytes()).hexdigest(),
    "clean_checkpoint_sha256": hashlib.sha256(Path("artifacts/runs/clean/model.safetensors").read_bytes()).hexdigest(),
}
run_context_path = export_dir / "artifacts/metrics/run_context.json"
run_context_path.parent.mkdir(parents=True, exist_ok=True)
run_context_path.write_text(json.dumps(run_context, indent=2) + "\n", encoding="utf-8")
pip_freeze_path = export_dir / "artifacts/metrics/pip_freeze.txt"
pip_freeze_path.write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")

archive = shutil.make_archive("/kaggle/working/hackathon_export", "zip", export_dir)
print("Validated export ready in Kaggle Output:", archive)